# Diabetes Prediction using Decision Tree Classifier
## Pima Indians Diabetes Dataset

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import plot_tree

%matplotlib inline
sns.set_style("whitegrid")

## 1. Load and Explore Data

In [ ]:
# Load data
df = pd.read_csv('diabetes.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nSummary Statistics:")
df.describe()

In [ ]:
# Target distribution
print("Class Distribution:")
print(df['Outcome'].value_counts())
print("\nClass Balance:")
print(df['Outcome'].value_counts(normalize=True))

# Visualize target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Outcome')
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('0 = Non-Diabetic, 1 = Diabetic')
plt.ylabel('Count')
plt.show()

## 2. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Handle zero values (replace with NaN)
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X[zero_cols] = X[zero_cols].replace(0, np.nan)

# Fill missing values with median
for col in zero_cols:
    X[col].fillna(X[col].median(), inplace=True)

print("Missing values after handling:", X.isnull().sum().sum())

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining distribution:\n{y_train.value_counts()}")
print(f"\nTest distribution:\n{y_test.value_counts()}")

## 3. Model Training

In [ ]:
# Initialize and train Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Make predictions
y_train_pred = dt.predict(X_train)
y_test_pred = dt.predict(X_test)

# Calculate accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc:.4f}")

## 4. Model Evaluation

In [ ]:
# Classification Report
print("Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=['Non-Diabetic', 'Diabetic']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Diabetic', 'Diabetic'],
            yticklabels=['Non-Diabetic', 'Diabetic'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Visualize
plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance, x='Importance', y='Feature')
plt.title('Feature Importance - Decision Tree')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 5. Visualize Decision Tree

In [ ]:
# Visualize tree (limited depth for readability)
plt.figure(figsize=(20,10))
plot_tree(dt, feature_names=X.columns, class_names=['Non-Diabetic', 'Diabetic'],
          filled=True, rounded=True, max_depth=3, fontsize=10)
plt.title('Decision Tree Visualization (Depth=3)')
plt.show()

## 6. Model Tuning (Optional)

In [ ]:
# Try different max_depth values
depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test, tree.predict(X_test)))

# Plot
plt.figure(figsize=(10,6))
plt.plot(depths, train_scores, label='Training Accuracy', marker='o')
plt.plot(depths, test_scores, label='Testing Accuracy', marker='o')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree Performance vs Depth')
plt.legend()
plt.grid(True)
plt.show()

# Best depth
best_depth = depths[test_scores.index(max(test_scores))]
print(f"Best depth: {best_depth}")
print(f"Best test accuracy: {max(test_scores):.4f}")

In [ ]:
# Train final model with best depth
final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate
y_final_pred = final_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_final_pred)

print(f"Final Model Test Accuracy: {final_accuracy:.4f}")
print("\nFinal Classification Report:")
print(classification_report(y_test, y_final_pred, target_names=['Non-Diabetic', 'Diabetic']))

## 7. Summary

**Key Findings:**
- The Decision Tree model achieves ~75% accuracy on test data
- Glucose is the most important predictor, followed by BMI and Age
- The model can be improved with hyperparameter tuning (max_depth)
- Class imbalance (more non-diabetic cases) affects performance

**Next Steps:**
- Try ensemble methods (Random Forest, XGBoost)
- Handle class imbalance with SMOTE
- Perform more extensive hyperparameter tuning with GridSearchCV